# Model 8 - Transfer Learning + Augmentation (MobileNetV2)

Same frozen-MobileNetV2 model as model 7, but trained on the **augmented**
pipeline (`augment=True`: flip/rotate/shift/zoom on the train set only). Tests
whether augmentation pushes the 0.94 transfer model any higher - the frozen
ImageNet features are robust, so augmentation may help here where it hurt the
small from-scratch GAP model.

## 1. Setup & imports

In [1]:
import os
import sys

sys.path.append(os.path.join(os.getcwd(), "preprocessing", "label_mapping"))
sys.path.append(os.path.join(os.getcwd(), "preprocessing", "data_loader"))
sys.path.append(os.path.join(os.getcwd(), "model"))
sys.path.append(os.path.join(os.getcwd(), "evaluation", "model_metrics"))
sys.path.append(os.path.join(os.getcwd(), "evaluation", "plots"))

import pandas as pd
from tensorflow.keras.callbacks import EarlyStopping

from label_mapping import build_labeled_dataset
from data_loader import build_train_val_test_generators
from transfer_mobilenet import build_transfer_mobilenet
from model_metrics import debug_model, evaluate_model, record_result
from plots import plot_misclassified

## 2. Load & label the dataset

In [2]:
df = build_labeled_dataset()
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 26179 entries, 0 to 26178
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   image_path  26179 non-null  str  
 1   label_it    26179 non-null  str  
 2   label_en    26179 non-null  str  
dtypes: str(3)
memory usage: 613.7 KB


## 3. Preprocess: split, resize, normalize, **augment**

Identical split/size to every model, but `augment=True` adds random
flip/rotate/shift/zoom to the **train** generator only (val/test stay clean).

In [3]:
train_generator, val_generator, test_generator = build_train_val_test_generators(
    df, project_root=os.getcwd(), image_size=(128, 128), augment=True
)
train_generator.class_indices

Found 18325 validated image filenames belonging to 10 classes.


Found 3927 validated image filenames belonging to 10 classes.


Found 3927 validated image filenames belonging to 10 classes.


{'butterfly': 0,
 'cat': 1,
 'chicken': 2,
 'cow': 3,
 'dog': 4,
 'elephant': 5,
 'horse': 6,
 'sheep': 7,
 'spider': 8,
 'squirrel': 9}

## 4. Build the model

Same `build_transfer_mobilenet` as model 7 (frozen MobileNetV2 + GAP head).

In [4]:
model = build_transfer_mobilenet(
    input_shape=train_generator.image_shape,
    num_classes=len(train_generator.class_indices),
)
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling (Rescaling)           │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_128            │ (None, 4, 4, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 10)             │        12,810 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,270,794 (8.66 MB)

 Trainable params: 12,810 (50.04 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

## 5. Train - or load a saved model

Saves to a **separate** file so model 7's winner stays intact. Set
`RETRAIN = True` to force a fresh run.

In [5]:
import time
from tensorflow import keras

MODEL_PATH = "models_saved/transfer_mobilenet_aug.keras"
RETRAIN = False

if not RETRAIN and os.path.exists(MODEL_PATH):
    model = keras.models.load_model(MODEL_PATH)
    history, train_time_min = None, None
    print(f"Loaded {MODEL_PATH} (skipped training).")
else:
    early_stopping = EarlyStopping(
        monitor="val_loss", patience=3, restore_best_weights=True
    )
    start_time = time.time()
    history = model.fit(
        train_generator,
        validation_data=val_generator,
        epochs=20,
        callbacks=[early_stopping],
    )
    train_time_min = round((time.time() - start_time) / 60, 1)
    print(f"Trained in {train_time_min} min.")

Epoch 1/20


  1/573 ━━━━━━━━━━━━━━━━━━━━ 18:28 2s/step - accuracy: 0.1250 - loss: 3.3101

  3/573 ━━━━━━━━━━━━━━━━━━━━ 38s 67ms/step - accuracy: 0.1354 - loss: 3.2702

  4/573 ━━━━━━━━━━━━━━━━━━━━ 38s 68ms/step - accuracy: 0.1406 - loss: 3.1785

  5/573 ━━━━━━━━━━━━━━━━━━━━ 41s 73ms/step - accuracy: 0.1562 - loss: 3.0876

...

571/573 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step - accuracy: 0.9032 - loss: 0.3220

572/573 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step - accuracy: 0.9032 - loss: 0.3220

573/573 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step - accuracy: 0.9032 - loss: 0.3218

573/573 ━━━━━━━━━━━━━━━━━━━━ 56s 98ms/step - accuracy: 0.9032 - loss: 0.3218 - val_accuracy: 0.9277 - val_loss: 0.2438


Trained in 4.5 min.


## 6. Training history

In [6]:
if history is not None:
    history_df = pd.DataFrame(history.history)
    history_df.index.name = "epoch"
    history_df["acc_gap"] = history_df["accuracy"] - history_df["val_accuracy"]
    display(history_df)
else:
    print("Model was loaded from disk - no training history this run.")

,accuracy,loss,val_accuracy,val_loss,acc_gap
epoch,,,,,
0,0.810477,0.611557,0.920550,0.248467,-0.110073
1,0.890259,0.362924,0.925643,0.233796,-0.035384
2,0.893042,0.346577,0.928699,0.239014,-0.035657
3,0.899536,0.333685,0.928189,0.236186,-0.028653
4,0.903247,0.321822,0.927680,0.243814,-0.024433


## 7. Evaluate: per-class report + confusion matrix

In [7]:
debug_model(model, val_generator)

  1/123 ━━━━━━━━━━━━━━━━━━━━ 1:01 502ms/step

  3/123 ━━━━━━━━━━━━━━━━━━━━ 5s 48ms/step   

  5/123 ━━━━━━━━━━━━━━━━━━━━ 5s 48ms/step

  7/123 ━━━━━━━━━━━━━━━━━━━━ 5s 48ms/step

  9/123 ━━━━━━━━━━━━━━━━━━━━ 5s 48ms/step

...

123/123 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step

123/123 ━━━━━━━━━━━━━━━━━━━━ 7s 52ms/step


              precision    recall  f1-score   support

   butterfly       0.93      0.97      0.95       317
         cat       0.90      0.94      0.92       250
     chicken       0.96      0.96      0.96       464
         cow       0.80      0.87      0.83       280
         dog       0.95      0.90      0.92       730
    elephant       0.93      0.91      0.92       217
       horse       0.87      0.93      0.90       393
       sheep       0.87      0.84      0.85       273
      spider       0.98      0.98      0.98       723
    squirrel       0.98      0.91      0.94       280

    accuracy                           0.93      3927
   macro avg       0.92      0.92      0.92      3927
weighted avg       0.93      0.93      0.93      3927



C:\Users\GBT H610M-\Documents\ironHack\task\project-2\evaluation\model_metrics\model_metrics.py:92: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,butterfly,cat,chicken,cow,dog,elephant,horse,sheep,spider,squirrel
butterfly,307,0,1,0,1,0,0,0,8,0
cat,1,234,3,0,7,1,2,0,0,2
chicken,5,2,445,1,5,1,2,1,2,0
cow,0,0,0,244,5,3,15,11,1,1
dog,2,17,8,12,654,2,26,7,1,1
elephant,0,0,0,5,5,198,2,5,2,0
horse,1,0,1,13,4,1,364,8,0,1
sheep,0,0,2,28,2,5,6,229,1,0
spider,12,1,2,0,1,0,0,0,706,1
squirrel,2,6,2,2,7,1,0,3,3,254


## 8. Misclassified images

In [8]:
plot_misclassified(model, val_generator)

  1/123 ━━━━━━━━━━━━━━━━━━━━ 9s 76ms/step

  2/123 ━━━━━━━━━━━━━━━━━━━━ 6s 53ms/step

  4/123 ━━━━━━━━━━━━━━━━━━━━ 5s 49ms/step

  6/123 ━━━━━━━━━━━━━━━━━━━━ 5s 48ms/step

  8/123 ━━━━━━━━━━━━━━━━━━━━ 5s 48ms/step

...

119/123 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step

121/123 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step

123/123 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step

123/123 ━━━━━━━━━━━━━━━━━━━━ 6s 49ms/step


C:\Users\GBT H610M-\Documents\ironHack\task\project-2\evaluation\plots\plots.py:74: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. Record result to the tracking sheet

In [9]:
learning_rate = round(float(model.optimizer.learning_rate.numpy()), 6)

row = evaluate_model(model, val_generator, "transfer_mobilenet_aug")
row.update({
    "architecture": "MobileNetV2 (ImageNet, frozen) -> GlobalAveragePooling -> Dropout(0.5) + augmentation",
    "learning_rate": learning_rate,
    "train_time_min": train_time_min,
    "notes": "Transfer learning + train augmentation (val metrics). Does augmentation beat plain transfer (model 7)?",
})
record_result(row)

  1/123 ━━━━━━━━━━━━━━━━━━━━ 9s 79ms/step

  3/123 ━━━━━━━━━━━━━━━━━━━━ 5s 46ms/step

  5/123 ━━━━━━━━━━━━━━━━━━━━ 5s 46ms/step

  7/123 ━━━━━━━━━━━━━━━━━━━━ 5s 46ms/step

  9/123 ━━━━━━━━━━━━━━━━━━━━ 5s 46ms/step

...

119/123 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step

121/123 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step

123/123 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step

123/123 ━━━━━━━━━━━━━━━━━━━━ 6s 48ms/step


,model,architecture,learning_rate,train_time_min,accuracy,precision,recall,f1,notes
0,baseline_cnn,3 conv blocks (6 Conv) -> Flatten -> Dropout(0.5),0.001,15.6,0.6988,0.6875,0.6569,0.6677,Baseline (val metrics). Overfits (~13% train-v...
1,gap_cnn,3 conv blocks (6 Conv) -> GlobalAveragePooling...,0.001,32.3,0.7403,0.7321,0.7111,0.7136,Flatten -> GAP (val metrics). Less overfit (~8...
2,gap_bn_cnn,3 conv blocks (Conv-BN-ReLU) -> GlobalAverageP...,0.001,62.6,0.6924,0.7526,0.6448,0.6593,GAP + BatchNorm (val metrics). Test if BN beat...
3,gap_aug,3 conv blocks (6 Conv) -> GlobalAveragePooling...,0.001,47.3,0.6677,0.6511,0.6433,0.6300,GAP + train augmentation (flip/rotate/shift/zo...
4,gap_aug2,3 conv blocks (6 Conv) -> GlobalAveragePooling...,0.001,54.7,0.7179,0.7065,0.7053,0.6895,Augmentation retry: dropout 0.3 + 60 epochs. T...
5,gap_deep,"4 conv blocks (8 Conv, +256) -> GlobalAverageP...",0.001,14.6,0.1859,0.0186,0.1000,0.0314,FAILED to train: plain deep CNN collapsed to ~...
6,transfer_mobilenet,"MobileNetV2 (ImageNet, frozen) -> GlobalAverag...",0.001,4.1,0.9404,0.9338,0.9309,0.9321,Transfer learning (val metrics). Frozen ImageN...
7,transfer_mobilenet_aug,"MobileNetV2 (ImageNet, frozen) -> GlobalAverag...",0.001,4.5,0.9256,0.9162,0.9192,0.9172,Transfer learning + train augmentation (val me...


## 10. Save the model

In [10]:
if history is not None:
    os.makedirs("models_saved", exist_ok=True)
    model.save(MODEL_PATH)
    print(f"Saved {MODEL_PATH}")
else:
    print(f"Using existing {MODEL_PATH}")

Saved models_saved/transfer_mobilenet_aug.keras


## 11. Compare all models

In [11]:
from model_metrics import plot_accuracy_comparison

plot_accuracy_comparison()

C:\Users\GBT H610M-\Documents\ironHack\task\project-2\evaluation\model_metrics\model_metrics.py:109: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,model,architecture,learning_rate,train_time_min,accuracy,precision,recall,f1,notes
5,gap_deep,"4 conv blocks (8 Conv, +256) -> GlobalAverageP...",0.001,14.6,0.1859,0.0186,0.1000,0.0314,FAILED to train: plain deep CNN collapsed to ~...
3,gap_aug,3 conv blocks (6 Conv) -> GlobalAveragePooling...,0.001,47.3,0.6677,0.6511,0.6433,0.6300,GAP + train augmentation (flip/rotate/shift/zo...
2,gap_bn_cnn,3 conv blocks (Conv-BN-ReLU) -> GlobalAverageP...,0.001,62.6,0.6924,0.7526,0.6448,0.6593,GAP + BatchNorm (val metrics). Test if BN beat...
0,baseline_cnn,3 conv blocks (6 Conv) -> Flatten -> Dropout(0.5),0.001,15.6,0.6988,0.6875,0.6569,0.6677,Baseline (val metrics). Overfits (~13% train-v...
4,gap_aug2,3 conv blocks (6 Conv) -> GlobalAveragePooling...,0.001,54.7,0.7179,0.7065,0.7053,0.6895,Augmentation retry: dropout 0.3 + 60 epochs. T...
1,gap_cnn,3 conv blocks (6 Conv) -> GlobalAveragePooling...,0.001,32.3,0.7403,0.7321,0.7111,0.7136,Flatten -> GAP (val metrics). Less overfit (~8...
7,transfer_mobilenet_aug,"MobileNetV2 (ImageNet, frozen) -> GlobalAverag...",0.001,4.5,0.9256,0.9162,0.9192,0.9172,Transfer learning + train augmentation (val me...
6,transfer_mobilenet,"MobileNetV2 (ImageNet, frozen) -> GlobalAverag...",0.001,4.1,0.9404,0.9338,0.9309,0.9321,Transfer learning (val metrics). Frozen ImageN...
